In [ ]:
import kagglehub
import shutil
import os
import glob

# Scarica il dataset
path = kagglehub.dataset_download("tunguz/used-car-auction-prices")

# Definisci la cartella di destinazione
destination_folder = "../dataset"

# Crea la cartella se non esiste
os.makedirs(destination_folder, exist_ok=True)

# Trova tutti i file CSV nella cartella scaricata
csv_files = glob.glob(os.path.join(path, "*.csv"))

# Sposta solo i file CSV nella cartella desiderata
for file in csv_files:
    shutil.move(file, destination_folder)

# Opzionale: Rimuove la cartella originale se è vuota
if not os.listdir(path):
    os.rmdir(path)

print(f"Dataset spostato in: {destination_folder}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, numpy as np


diagrams_dir = "../../diagrams"
df = pd.read_csv('../../dataset/car_prices.csv', on_bad_lines="skip")
df

In [ ]:
df.info()

In [ ]:
df.describe(include='number').transpose()

In [ ]:
df.isnull().sum()

missing_values = df.isna().sum()
missing_values = missing_values[missing_values > 0]

if not missing_values.empty:
    plt.figure(figsize=(10, 6))
    missing_values.plot(kind='bar', color='skyblue', edgecolor='black')
    plt.title('Valori mancanti per colonna', fontsize=16)
    plt.xlabel('Colonne', fontsize=14)
    plt.ylabel('Numero di valori mancanti', fontsize=14)
    plt.tight_layout()

    plt.savefig(f'{diagrams_dir}/missing_values_bar_chart.svg')
    print("Grafico salvato come 'missing_values_bar_chart.svg'")
else:
    print("Non ci sono valori mancanti nel dataset.")

In [ ]:
def plot_distribution(df, column_name, labels, bins, palette='Blues', right=False, output_dir=diagrams_dir):
    """
    Crea una nuova feature categorizzando i valori di una colonna in intervalli basati sui quantili e genera un istogramma della distribuzione.

    Args:
        df (pd.DataFrame): DataFrame contenente la colonna da analizzare.
        column_name (str): Nome della colonna su cui basare il plot.
        bins (list): Lista dei limiti per i bin.
        labels (list): Lista delle etichette da associare ai bin.
        palette (str, optional): Palette di colori per il plot.
        right (bool, optional): Determina se l'intervallo include il limite destro.
        output_dir (str, optional): Directory in cui salvare il grafico.

    Returns:
        pd.DataFrame: DataFrame con la nuova feature 'value_range'.
    """
    df_tmp = df.copy()

    # Creazione della nuova feature categorizzata
    df_tmp[f'{column_name}_range'] = pd.cut(df_tmp[column_name], bins=bins, labels=labels, right=right)

    # Creazione del grafico
    plt.figure(figsize=(12, 6), facecolor='white')  # Imposta lo sfondo della figura a bianco
    ax = plt.gca()
    ax.set_facecolor('white')  # Imposta lo sfondo dell'area dell'asse a bianco

    # Crea il countplot
    sns.countplot(x=f'{column_name}_range', data=df_tmp, hue=f'{column_name}_range',
                  palette=palette, order=labels, legend=False)

    # Imposta il titolo e le etichette
    plt.title(f'Distribuzione dei {column_name}', fontsize=16)
    plt.xlabel(f'Intervallo di {column_name}', fontsize=12)
    plt.xticks(fontsize=10)
    plt.ylabel('Frequenza', fontsize=12)
    plt.grid(axis='y', alpha=0.3)

    # Imposta il bordo dell'asse su nero
    for spine in ax.spines.values():
        spine.set_edgecolor('black')

    # Salvataggio del grafico
    os.makedirs(output_dir, exist_ok=True)
    plot_path = os.path.join(output_dir, f'{column_name}_distribution.svg')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Grafico salvato in {plot_path}")

In [ ]:
# confini di ciascun intervallo per il prezzo
price_bins = [0, 10000, 20000, 30000, 40000, 50000, 100000, 230000]
# etichette degli intervalli per il prezzo
price_labels = ["0k-10k", "10k-20k", "20k-30k", "30k-40k", "40k-50k","50k-100k","100k-230k"]
plot_distribution(df, 'sellingprice', price_labels, price_bins)

In [ ]:
condition_bins = [0, 1, 2, 3, 4, 5]
condition_labels = ["0-1", "1-2", "2-3", "3-4", "4-5"]
plot_distribution(df, 'condition', condition_labels, condition_bins, 'Greens', True)

In [ ]:
year_bins = [1982, 1990, 2000, 2010, 2015]
year_labels = ["1982-1990", "1990-2000", "2000-2010", "2010-2015"]
plot_distribution(df, 'year', year_labels, year_bins)

In [ ]:
odometer_bins = [1, 10000, 25000, 50000, 75000, 100000, 125000, 150000, 175000, 200000, 225000, 250000, 275000, 300000]
odomete_labels = ['1-10000', '10000 - 25000', '25000 - 50000', '50000 - 75000', '75000 - 100000', '100000 - 125000', '125000 - 150000', '150000 - 175000', '175000 - 200000', '200000 - 225000', '225000 - 250000', '250000 - 275000', '275000 - 300000']
plot_distribution(df, 'odometer', odomete_labels, odometer_bins, 'Reds')

In [ ]:
df.duplicated().sum()

In [ ]:
def plot_line_chart(df, x_column, y_column='sellingprice', output_dir="../diagrams"):
    """
    Crea un grafico a linee tra due variabili (una sul lato x e una sul lato y) e salva l'immagine.

    Args:
        df (pd.DataFrame): DataFrame contenente le colonne da plottare.
        x_column (str): Nome della colonna da utilizzare sull'asse X.
        y_column (str): Nome della colonna da utilizzare sull'asse Y.
        output_dir (str, optional): Directory in cui salvare il grafico.

    Returns:
        None
    """
    # Creazione del grafico
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=x_column, y=y_column, data=df, color='black', marker='o', linewidth=2, label=y_column)

    # Titolo e etichette
    plt.title(f'{y_column} vs. {x_column}', fontsize=16)
    plt.xlabel(x_column.capitalize(), fontsize=12)
    plt.ylabel(y_column.capitalize(), fontsize=12)
    plt.grid(alpha=0.3)
    plt.legend()

    # Salvataggio del grafico
    os.makedirs(output_dir, exist_ok=True)
    plot_path = os.path.join(output_dir, f'{x_column}_vs_{y_column}_line_plot.svg')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Grafico salvato in {plot_path}")

In [ ]:
plot_line_chart(df, 'year')

In [ ]:
plot_line_chart(df, 'condition')

In [ ]:
plot_line_chart(df, 'odometer')

In [ ]:
plot_line_chart(df, 'mmr')

In [ ]:
# Selezione dinamica delle caratteristiche numeriche
numerical_features = df.select_dtypes(include=['number']).columns.tolist()

# Calcolo della matrice di correlazione
corr_matrix = df[numerical_features].corr()

# Creazione della heatmap
plt.figure(figsize=(15, 12))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, cbar=True)
plt.title('Matrice di correlazione delle caratteristiche numeriche', fontsize=16)

os.makedirs(diagrams_dir, exist_ok=True)
path = f"{diagrams_dir}/correlation_matrix.svg"
plt.savefig(path, dpi=300, bbox_inches='tight')
plt.close()

print(f"Matrice di correlazione salvata in {path}")